# Feature Selection

Feature selection is the process of determining what features to include in your model. Why do we need feature selection?

*   To prevent overfitting - models with more features are more prone to overfitting
*   To simplify our model - simpler models are easier to interpret and debug
*   To reduce computation time - models run faster with less features

There are several ways we can perform feature selection
1. Filter methods
2. Wrapper methods such as forward selection (there are known issues with some of these methods)
3. Regularization methods (these are discussed in more detail in another lecture)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler


## Read in Data

In [ ]:
# Read in data
url = "http://ddc-datascience.s3.amazonaws.com/CarPrice_Assignment.csv"
cars = pd.read_csv( url )
cars.head()

In [ ]:
cars.shape

In [ ]:
cars.info()

In [ ]:
cars.describe().transpose()

## Data Cleaning

In [ ]:
cars["price"].sort_values(ascending = False)

In [ ]:
cars.nunique().sort_values(ascending = False)

In [ ]:
# Drop car_ID column
cars.drop('car_ID', inplace = True, axis = 1)

In [ ]:
cars.columns.sort_values().to_list()

In [ ]:
# Look at CarName column
cars['CarName'].value_counts()

In [ ]:
# Keep just the name of the car manufacturer
car_split = lambda x: x.split(" ")[0]

def car_split(x):
  return x.split(" ")[0]

cars['carCompany'] = cars['CarName'].apply(lambda x: x.split(" ")[0])

# pd.Series([ x.split(" ")[0] for x in cars['CarName'] ][:10])

# cars['CarName'].apply(lambda x: x.split(" ")[0])


In [ ]:
cars['carCompany'].value_counts()

In [ ]:
# # Fix spelling
# cars['carCompany'] = cars['carCompany'].str.replace('vw', 'volkswagen')
# cars['carCompany'] = cars['carCompany'].str.replace('vokswagen', 'volkswagen')
# cars['carCompany'] = cars['carCompany'].str.replace('porcshce', 'porsche')
# cars['carCompany'] = cars['carCompany'].str.replace('maxda', 'mazda')
# cars['carCompany'] = cars['carCompany'].str.replace('toyouta', 'toyota')
# cars['carCompany'] = cars['carCompany'].str.replace('Nissan', 'nissan')


In [ ]:
# create mapping between incorrect and correct spelling
spelling_dict={
  'alfa-romero': 'alfa-romeo',
  'maxda'      : 'mazda',
  'Nissan'     : 'nissan',
  'porcshce'   : 'porsche',
  'toyouta'    : 'toyota',
  'vw'         : 'volkswagen',
  'vokswagen'  : 'volkswagen',
}

In [ ]:
# Fix spelling
for k, v in spelling_dict.items():
  cars['carCompany'] = cars['carCompany'].str.replace( k, v )

cars['carCompany'].value_counts()


In [ ]:
# Drop carname column
cars.drop('CarName', axis = 1, inplace = True)

In [ ]:
cars.head()


In [ ]:
display(cars['doornumber'].value_counts())
display(cars['cylindernumber'].value_counts())


In [ ]:
# Change doornumber and cylindernumber to numeric form
text_to_number = {
  'two'   :  "2",
  'three' :  "3",
  'four'  :  "4",
  'five'  :  "5",
  'six'   :  "6",
  'eight' :  "8",
  'twelve': "12",
}

cars['doornumber'] = cars['doornumber'].replace( text_to_number ).astype(int)
cars['cylindernumber'] = cars['cylindernumber'].replace( text_to_number ).astype(int)

cars[['doornumber','cylindernumber']].head()


In [ ]:
cars[['doornumber','cylindernumber']].head().transpose()


In [ ]:
cars.dtypes.sort_values()

In [ ]:
# Pull out object data types
cars_categorical = cars.select_dtypes(include = ['object'])
cars_categorical.head()

In [ ]:
for col in cars_categorical.columns:
  print(col, cars_categorical[col].unique())

In [ ]:
cars_categorical.nunique()


In [ ]:
cars_categorical.nunique().sum()


### One-hot encode

In [ ]:
# One hot encode these categories
cars_encoded = pd.get_dummies(cars_categorical, dtype='int')
cars_encoded.head()

In [ ]:
cars_encoded.shape

In [ ]:
cars_categorical.shape

In [ ]:
# Combine encoded dataframe to data frame with just numeric values
cars_numerical = cars.select_dtypes(include = ['int64', 'float64'])
cars_numerical.shape

In [ ]:
cars_combined = pd.concat([cars_numerical, cars_encoded], axis = 1)
cars_combined.head()

In [ ]:
cars_combined.shape

In [ ]:
cars_combined.dtypes.value_counts()

## Processing

First, we will fit a model with all predictors.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split


In [ ]:
# split into Target and Features
X = cars_combined.drop('price', axis = 1)
y = cars_combined['price']

In [ ]:
# Perform CV
n = 500
results = np.zeros(n)
for idx in range(n):
  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25)
  model = LinearRegression()
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  results[idx] = np.sqrt(mean_squared_error(y_test,y_pred))

print(f"CV RMSE: {results.mean().round(2)}")
print(f"Number of Predictors: {len(X.columns)}")

## Feature Selection

## Filter Methods
With filter methods, we will calculate a statistical metric for each column. Based on that metric, it will be decided whether that feature will be kept or removed from our predictive model. *Note*: the choice of statistical metric depends on the data type of the feature.

**Pros**: Easy to implement

**Cons**: Does not account for interactions between features

Image from [Toward Data Science:](https://towardsdatascience.com/learn-how-to-do-feature-selection-the-right-way-61bca8557bef)

![An image](https://miro.medium.com/max/1400/1*tzfWABEHK9-4SOaSl1mdRA.png)


### Numerical Predictors & Numerical Response - Pearson Correlation Coefficient

In [ ]:
# Look at numerical predictors
cars_numerical.head()

In [ ]:
cars_numerical.shape

In [ ]:
# Sort correlations w/ price
cars_numerical.corr()['price'].abs().sort_values(ascending = False)

#### Correlation bar graph


In [ ]:
# Plot result
cars_numerical.corr()['price'].abs().sort_values(ascending = False).plot(kind = 'bar', figsize = (10,5)) ;

WARNING: The correlation coefficient only measures the strength of the linear relationship between two variables. If the relationship is nonlinear, it may not be reflected in the correlation coefficient.

Example from [Toward Data Science:](https://towardsdatascience.com/learn-how-to-do-feature-selection-the-right-way-61bca8557bef)
![An image](https://miro.medium.com/max/1400/1*cY24YPkTGmuzaSaUmdTbmw.png)

In [ ]:
# Fit linear regression model w/ just continuous predictors highly correlated with response
corrs = cars_numerical.corr()['price'].abs().sort_values(ascending = False)
keep = corrs[(corrs>.2) & (corrs <1)]
keep

In [ ]:
X_corr = X[keep.index]
X_corr.head()

In [ ]:
X_corr.shape

In [ ]:
# Combine correlated numerical predictors with all encoded categorical predictors
X = pd.concat([X_corr, cars_encoded], axis = 1)
X.head()

In [ ]:
X.shape

In [ ]:
# Perform CV
n = 500
results = np.zeros(n)
for idx in range(n):
  X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25)
  model = LinearRegression()
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  results[idx] = np.sqrt(mean_squared_error(y_test,y_pred))

print(f"CV RMSE: {results.mean().round(2)}")
print(f"Number of Predictors: {len(X.columns)}")

### ANOVA - Categorical Predictors & Numerical Response
ANOVA works by testing if the means of different categories are statistically different. For example, it would test to see if the mean price for BMWs is statistically different from the mean price of Buicks.

It is based on the following two hypotheses
H0: Means of all groups are equal. (null hypothesis)
H1: At least one mean of the groups are different.

ANOVA calculates an F-Test for statistical signficance. The larger the value of the F-test, the more likely we are to reject the null hypothesis.

In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression

fvalue_selector = SelectKBest(f_regression, k='all')  #select features best ANOVA F-Values
fvalue_selector.fit_transform(cars_encoded, y)

# Plot results
results = pd.DataFrame({'ANOVA': fvalue_selector.scores_}, index = cars_encoded.columns)
results.sort_values(by = 'ANOVA', ascending = False).plot(kind = 'bar', figsize = (15,5))

In [ ]:
# Just keep the results with F-values greater than 20
anova = results.sort_values(by = 'ANOVA', ascending = False)
keep = anova['ANOVA'][anova['ANOVA'] > 10]
keep

In [ ]:
keep.index.size


In [ ]:
X_anova = X[keep.index]

In [ ]:
X_anova

In [ ]:
X2 = pd.concat([X_anova, X_corr], axis = 1)
X2.head()

In [ ]:
( X_corr.shape, X_anova.shape, X.shape, X2.shape)

In [ ]:
# Perform CV
n = 500
rmse_results = np.zeros(n)
for idx in range(n):
  X_train, X_test, y_train, y_test = train_test_split(X2,y,test_size=0.25)
  model = LinearRegression()
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  rmse_results[idx] = np.sqrt(mean_squared_error(y_test,y_pred))
print(f"CV RMSE: {rmse_results.mean().round(2)}")
print(f"Number of Predictors: {len(X2.columns)}")

## Regularization

In [ ]:
from sklearn.linear_model import Lasso

Scale the data frame using the Standard Scaler

In [ ]:
cars_combined

In [ ]:
cars_combined_bak = cars_combined.copy()
cars_combined_bak.shape


In [ ]:
# restore
cars_combined = cars_combined_bak.copy()
cars_combined


Use standard scaler to scale data set.

This data set is the cleaned without the feature selection from correlation.

In [ ]:
# %%capture --no-stdout
X_lasso = cars_combined.drop('price', axis = 1)
y_lasso = cars_combined['price']

# Specify values of alpha we want to try
alpha = [50, 75, 100, 200, 300, ]

# Create empty list to hold results
results_alpha = []

for a in alpha:
  # Calculate CV RMSE
  n = 200
  results = np.zeros(n)

  for idx in range(n):
    # Train, test split
    X_train, X_test, y_train, y_test = train_test_split(X_lasso, y_lasso, test_size=0.25)

    # Standard scale only training features
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train = scaler.transform(X_train)

    # Scale testing features
    X_test = scaler.transform(X_test)

    # Create the model
    my_fit_lasso = Lasso(alpha = a, tol = .01, max_iter=10_000); # Looping through alpha

    # Fit regression
    my_fit_lasso.fit(X_train, y_train)

    # Predict
    y_pred = my_fit_lasso.predict(X_test)

    # Calculate and record performance metric
    results[idx] = np.sqrt(mean_squared_error(y_test,y_pred))

  results_alpha.append(results.mean())

results_df = pd.DataFrame({'Alpha': alpha, 'RMSE': results_alpha})


In [ ]:
results_df.sort_values( by = 'RMSE' )


Get the Alpha that results in the lowest RMSE.

In [ ]:
results_df.sort_values( by = 'RMSE' )["Alpha"][0]


In [ ]:
results_df.sort_values( by = 'RMSE' )["Alpha"].iloc[0]


In [ ]:
alpha = results_df.sort_values( by = 'RMSE' )["Alpha"].iloc[0]
alpha

In [ ]:
# %%capture --no-stdout
# Perform CV
n = 500
results = np.zeros(n)

for idx in range(n):
  # Train, test split
  X_train, X_test, y_train, y_test = train_test_split(X_lasso, y_lasso, test_size=0.25)

  # Standard scale only training features
  scaler = StandardScaler()
  scaler.fit(X_train)
  X_train = scaler.transform(X_train)

  # Scale testing features
  X_test = scaler.transform(X_test)

  # Create the model
  model = Lasso(alpha = alpha, tol = .01, max_iter=10000)

  # Fit regression
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate and record performance metric
  results[idx] = np.sqrt(mean_squared_error(y_test,y_pred))

print(f"CV RMSE: {results.mean().round(2)}")
print(f"Number of Predictors: {sum(model.coef_ >0)}")

In [ ]:
X_lasso.shape

In [ ]:
len(X_lasso.columns)

In [ ]:
# Plot model coefficients
model_results = pd.DataFrame()
model_results['Predictor'] = X_lasso.columns
model_results['Lasso'] = model.coef_
model_results.sort_values( by = ["Lasso"], ascending = False ).plot(
    x = 'Predictor',
    kind = 'bar',
    figsize = (20,5)
    )
plt.grid()

In [ ]:
model_results['Lasso'].abs().sort_values()

In [ ]:
# Plot parameters with coefficients greater than 0
greater_0 = model_results[(model_results['Lasso'] > 500 ) | (model_results['Lasso'] < -500 )]
greater_0.sort_values( by = ["Lasso"], ascending = False ).plot(x = 'Predictor', kind = 'bar', figsize = (10,5))
plt.grid()

In [ ]:
X_lasso[greater_0["Predictor"]]

## Leaky data


In [ ]:
X_lasso2 = X_lasso.copy()
X_lasso2.shape


In [ ]:
X_lasso2["leaky"] = y_lasso
X_lasso2.shape


In [ ]:
X_lasso2

In [ ]:
# %%capture --no-stdout
# Perform CV
n = 500
results = np.zeros(n)

for idx in range(n):
  # Train, test split
  X_train, X_test, y_train, y_test = train_test_split(X_lasso2, y_lasso, test_size=0.25)

  # Standard scale only training features
  scaler = StandardScaler()
  scaler.fit(X_train)
  X_train = scaler.transform(X_train)

  # Scale testing features
  X_test = scaler.transform(X_test)

  # Create the model
  model = Lasso(alpha = alpha, tol = .01, max_iter=10000)

  # Fit regression
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate and record performance metric
  results[idx] = np.sqrt(mean_squared_error(y_test,y_pred))

print(f"CV RMSE: {results.mean().round(2)}")
print(f"Number of Predictors: {sum(model.coef_ >0)}")